In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch

# Transformer Architecture: Implementation

* This notebook shows how an autoregressive language model based on a Generative Pre-Trained (GPT) transformer architecture can be implemented in PyTorch.

* This model will be trained using the book [Frankenstein; or, The Modern Prometheus](https://www.gutenberg.org/cache/epub/84/pg84.txt) by Mary Wollstonecraft Shelley.



# Organizing the Dataset


* The following code downloads the book and stores it into a Python string.

In [ ]:
import requests

# Project Gutenberg has many books stored as text files (https://www.gutenberg.org/)
raw_text = requests.get('https://www.gutenberg.org/cache/epub/84/pg84.txt').text # Downloads and stores the text into a string
raw_text = raw_text.partition('*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN; OR, THE MODERN PROMETHEUS ***')[2] # Removes foreword
raw_text = raw_text.partition('*** END OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN; OR, THE MODERN PROMETHEUS ***')[0] # Removes afterword

* In the following code:
    * The function `preprocess` converts a string so that any sequence of non-letters is substituted by a whitespace and uppercase letters are substituted by corresponding lowercase letters.
    * The function `tokenize` converts a string into a list of tokens (in this case, characters).
    * The class `Vocabulary` implements a vocabulary.

In [ ]:
import re
import collections


def preprocess(text):
    text = re.sub('[^A-Za-z]+', ' ', text) # Substitutes any sequence of non-letters by a whitespace
    text = text.lower() # Converts uppercase letters to lowercase letters

    return text


def tokenize(text, use_chars):
    if use_chars: # One token for each character
        return list(text)
    else: # One token for each sequence of letters
        return text.split()


class Vocabulary:
    def __init__(self, tokens):
        counter = collections.Counter(tokens)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True) # List of pairs where each pair is composed of a token and its frequency. Sorted according to decreasing frequency.

        self.unknown = '?' # Represents an unknown token

        self.id_to_token = sorted([token for token, freq in self.token_freqs]) + [self.unknown] # Maps an index to a token
        self.token_to_id = {token: id for id, token in enumerate(self.id_to_token)} # Maps a token to an index

    def __len__(self):
        return len(self.id_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)): # If called with a single token
            return self.token_to_id.get(tokens, self.token_to_id[self.unknown])
        else: # If called with a list of tokens
            return [self.token_to_id.get(token, self.token_to_id[self.unknown]) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)): # If called with a single index
            return self.id_to_token[indices]
        else: # If called with a list of indices
            return [self.id_to_token[index] for index in indices]

* The following code converts the raw text into a sequence of numbers, each of which corresponds to a token (in this case, a character).

In [ ]:
text = preprocess(raw_text)
print(text[:38]) # Prints the first 38 characters

tokens = tokenize(text, use_chars=True)
print(tokens[:38])

vocab = Vocabulary(tokens)
print(vocab.id_to_token)

indices = vocab[tokens]
print(indices[:38])

tokens_from_indices = vocab.to_tokens(indices)
print(tokens_from_indices[:38])

 frankenstein or the modern prometheus
[' ', 'f', 'r', 'a', 'n', 'k', 'e', 'n', 's', 't', 'e', 'i', 'n', ' ', 'o', 'r', ' ', 't', 'h', 'e', ' ', 'm', 'o', 'd', 'e', 'r', 'n', ' ', 'p', 'r', 'o', 'm', 'e', 't', 'h', 'e', 'u', 's']
[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '?']
[0, 6, 18, 1, 14, 11, 5, 14, 19, 20, 5, 9, 14, 0, 15, 18, 0, 20, 8, 5, 0, 13, 15, 4, 5, 18, 14, 0, 16, 18, 15, 13, 5, 20, 8, 5, 21, 19]
[' ', 'f', 'r', 'a', 'n', 'k', 'e', 'n', 's', 't', 'e', 'i', 'n', ' ', 'o', 'r', ' ', 't', 'h', 'e', ' ', 'm', 'o', 'd', 'e', 'r', 'n', ' ', 'p', 'r', 'o', 'm', 'e', 't', 'h', 'e', 'u', 's']


* For a pre-defined batch size $n$ and a pre-defined number of steps $M$, the sequence of indices will be partitioned into subsequences of length $M$, which will be grouped into batches of size $n$.

* The following code implements random partitioning using a Python generator.

In [ ]:
def random_partitioning(sequence, batch_size, num_steps, offset=None):
    if offset is None:
        offset = np.random.randint(num_steps) # A random number between 0 and `num_steps` (excluding `num_steps`)

    sequence = sequence[offset:] # Discards the first `offset` elements of the sequence

    num_subseqs = (len(sequence) - 1) // num_steps # The number of subsequences of length `num_steps` that fit into the `sequence`

    subseq_indices = list(range(0, num_subseqs * num_steps, num_steps)) # A list with the index where each subsequence starts: [0, `num_steps`, 2 * `num_steps`, ...]

    np.random.shuffle(subseq_indices) # Randomizes the indices to enable creating batches composed of randomly chosen subsequences

    num_batches = num_subseqs // batch_size # The number of batches required to fit every subsequence

    for i in range(0, num_batches * batch_size, batch_size):
        batch_indices = subseq_indices[i: i + batch_size] # The index where each subsequence that should be part of this batch starts

        X = [sequence[j: j + num_steps] for j in batch_indices] # Organizes each subsequence into a row
        Y = [sequence[j + 1: j + 1 + num_steps] for j in batch_indices] # Organizes each subsequence (shifted by one index) into a row

        yield torch.tensor(X), torch.tensor(Y) # Yields a pair of `batch_size` x `num_steps` matrices

* The following code displays the first two batches of input subsequences and target subsequences (converted into batches of tokens).

In [ ]:
print('Partitioning and batching the sequence of token indices (first two batches, converted into batches of tokens):')
for i, (X, Y) in enumerate(random_partitioning(indices, batch_size=3, num_steps=16)):
    X_tokens = np.array([vocab.to_tokens(list(X[j])) for j in range(X.shape[0])])
    Y_tokens = np.array([vocab.to_tokens(list(Y[j])) for j in range(Y.shape[0])])
    print(f'X: \n{X_tokens}.')
    print(f'Y: \n{Y_tokens}.\n')
    print()

    if i >= 1:
        break

Partitioning and batching the sequence of token indices (first two batches, converted into batches of tokens):
X: 
[['w' 'o' 'r' 'd' 's' ' ' 'i' ' ' 'p' 'e' 'r' 'c' 'e' 'i' 'v' 'e']
 ['i' 's' 'm' ' ' 'a' 'n' 'd' ' ' 'h' 'i' 's' ' ' 'i' 'n' 's' 't']
 ['r' 'e' ' ' 't' 'h' 'e' ' ' 'l' 'e' 'a' 's' 't' ' ' 'p' 'a' 'i']].
Y: 
[['o' 'r' 'd' 's' ' ' 'i' ' ' 'p' 'e' 'r' 'c' 'e' 'i' 'v' 'e' 'd']
 ['s' 'm' ' ' 'a' 'n' 'd' ' ' 'h' 'i' 's' ' ' 'i' 'n' 's' 't' 'r']
 ['e' ' ' 't' 'h' 'e' ' ' 'l' 'e' 'a' 's' 't' ' ' 'p' 'a' 'i' 'n']].


X: 
[['h' 'e' ' ' 'g' 'r' 'e' 'a' 't' 'e' 's' 't' ' ' 'a' 'f' 'f' 'e']
 ['a' 'n' 'd' ' ' 'h' 'a' 'p' 'p' 'y' ' ' 'm' 'y' ' ' 'a' 'u' 'n']
 [' ' 'a' 'c' 't' ' ' 'i' 'n' ' ' 'h' 'e' 'r' ' ' 't' 'u' 'r' 'n']].
Y: 
[['e' ' ' 'g' 'r' 'e' 'a' 't' 'e' 's' 't' ' ' 'a' 'f' 'f' 'e' 'c']
 ['n' 'd' ' ' 'h' 'a' 'p' 'p' 'y' ' ' 'm' 'y' ' ' 'a' 'u' 'n' 't']
 ['a' 'c' 't' ' ' 'i' 'n' ' ' 'h' 'e' 'r' ' ' 't' 'u' 'r' 'n' ' ']].




# Self-Attention

* The following code implements a self-attention layer as a PyTorch module called `SelfAttention`.

* The method `forward` will receive an $n \times T \times d$ tensor called `X`, where $n$ is the batch size, $T$ is the number of time steps, and $d$ is the input dimension. If the tensor `X` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) input vector.

* The method `forward` will output an $n \times T \times h$ tensor called `H`, where $n$ is the batch size, $T$ is the number of time steps, and $h$ is the output dimension. If the tensor `H` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) output vector.

* In principle, self-attention is not limited by a maximum sequence length $M$. However, defining a maximum sequence length allows pre-computing the mask `upper_triangle`, which leads to a more efficient implementation.

In [ ]:
class SelfAttention(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, query_dimension, max_seq_length):
        super(SelfAttention, self).__init__()

        self.query_layer = torch.nn.Linear(input_dimension, query_dimension) # A linear layer to transform each input tensor into a query tensor
        self.key_layer = torch.nn.Linear(input_dimension, query_dimension) # A linear layer to transform each input tensor into a key tensor
        self.value_layer = torch.nn.Linear(input_dimension, output_dimension) # A linear layer to transform each input tensor into a value tensor

        upper_triangle = torch.triu(torch.ones((max_seq_length, max_seq_length), dtype=bool), diagonal=1) # A `max_seq_length` x `max_seq_length` mask
        self.register_buffer('upper_triangle', upper_triangle) # This tensor is not a parameter of this module but should be be stored in the same device as them

        self.scaling = 1 / (query_dimension**0.5) # The scaling factor applied to the dot product between queries and keys

    def forward(self, X):
        """ Keyword arguments:
        `X` -- a `batch_size` x `num_steps` x `input_dimension` tensor (list of input matrices)
        """
        Q = self.query_layer(X) # A `batch_size` x `num_steps` x `query_dimension` query tensor
        K = self.key_layer(X) # A `batch_size` x `num_steps` x `query_dimension` key tensor
        V = self.value_layer(X) #  A `batch_size` x `num_steps` x `output_dimension` value tensor

        S = Q.matmul(K.transpose(1, 2)) * self.scaling # A `batch_size` x `num_steps` x `num_steps` score tensor

        upper_triangle = self.upper_triangle[:S.shape[1], :S.shape[2]] # A `num_steps` x `num_steps` mask
        S = S.masked_fill(upper_triangle, float('-inf')) # A `batch_size` x `num_steps` x `num_steps` masked score tensor

        A = torch.nn.functional.softmax(S, dim=2) # A `batch_size` x `num_steps` x `num_steps` attention tensor

        H = A.matmul(V) # A `batch_size` x `num_steps` x `output_dimension` tensor

        return H

# Multi-Headed Self-Attention

* The following code implements a multi-headed self-attention layer as a PyTorch module called `MultiHeadedSelfAttention`.

* The method `forward` will receive an $n \times T \times d$ tensor called `X`, where $n$ is the batch size, $T$ is the number of time steps, and $d$ is the input dimension. If the tensor `X` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) input vector.

* The method `forward` will output an $n \times T \times o$ tensor called `H`, where $n$ is the batch size, $T$ is the number of time steps, and $o$ is the output dimension. If the tensor `H` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) output vector.


In [ ]:
class MultiHeadedSelfAttention(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, n_heads, h_output_dimension, h_query_dimension, max_seq_length):
        super(MultiHeadedSelfAttention, self).__init__()

        self.heads = torch.nn.ModuleList([SelfAttention(input_dimension, h_output_dimension, h_query_dimension, max_seq_length) for _ in range(n_heads)]) # List of self-attention layers (heads)
        self.output_layer = torch.nn.Linear(n_heads * h_output_dimension, output_dimension) # A linear layer to transform the concatenated outputs of the self-attention layers

    def forward(self, X):
        """ Keyword arguments:
        `X` -- a `batch_size` x `num_steps` x `input_dimension` tensor (list of input matrices)
        """
        head_outputs = torch.cat([head(X) for head in self.heads], dim=2) # A `batch_size` x `num_steps` x `n_heads * h_output_dimension` tensor
        H = self.output_layer(head_outputs) # A `batch_size` x `num_steps` x `output_dimension` tensor

        return H

# Transformer

* The following code implements a transformer layer as a PyTorch module called `Transformer`.

* The method `forward` will receive an $n \times T \times d$ tensor called `X`, where $n$ is the batch size, $T$ is the number of time steps, and $d$ is the input dimension. If the tensor `X` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) input vector.

* The method `forward` will output an $n \times T \times d$ tensor, where $n$ is the batch size, $T$ is the number of time steps, and $d$ is the input dimension. If this tensor is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) output vector.


In [ ]:
class Transformer(torch.nn.Module):
    def __init__(self, input_dimension, n_heads, max_seq_length):
        super(Transformer, self).__init__()

        if input_dimension % n_heads != 0:
            raise Exception("The number of heads must divide the input dimension.")

        self.ln1 = torch.nn.LayerNorm(input_dimension) # The first layer normalization

        output_dimension = input_dimension
        h_output_dimension = input_dimension // n_heads
        h_query_dimension = h_output_dimension
        self.mhsa = MultiHeadedSelfAttention(input_dimension, output_dimension, n_heads, h_output_dimension, h_query_dimension, max_seq_length) # The multi-headed self-attention layer

        self.ln2 = torch.nn.LayerNorm(input_dimension) # The second layer normalization

        # The first fullly connected layer
        self.fc1 = torch.nn.Linear(input_dimension, 4 * input_dimension)
        self.relu = torch.nn.ReLU()

        # The second fully connected layer
        self.fc2 = torch.nn.Linear(4 * input_dimension, input_dimension)

    def forward(self, X):
        """ Keyword arguments:
        `X` -- a `batch_size` x `num_steps` x `input_dimension` tensor (list of input matrices)
        """
        X = X + self.mhsa(self.ln1(X)) # A `batch_size` x `num_steps` x `input_dimension` tensor
        X = X + self.fc2(self.relu(self.fc1(self.ln2(X)))) # A `batch_size` x `num_steps` x `input_dimension` tensor

        return X

# Generative Pre-Trained Transformer (GPT) Language Model

* The following code implements an autoregressive language model based on a Generative Pre-Trained Transformer (GPT) architecture in a PyTorch module called `GPTLanguageModel`.

* The method `forward` receives an $n \times T$ token indices matrix called `X` , where $n$ is the batch size and $T$ is the number of time steps.
    * The elements of the matrix `X` are encoded by a token embedding layer called `token_encoding`, resulting in an $n \times T \times d$ tensor called `tokens_encoded`. If `tokens_encoded` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each matrix has one row for each time step, so that each row contains a (transposed) code vector for a token.
    * A position embedding layer called `position_encoding` is used to create a $T \times d$ matrix called `positions_encoded`. The $t$-th row of `positions_encoded` contains a (transposed) code vector for time step $t$, for every $t$.
    * The input tensor for the remaining layers is obtained by adding `tokens_encoded` and `positions_encoded`. If `tokens_encoded` is interpreted as a list of $n$ matrices, this involves adding `positions_encoded` to each of them. The resulting $n \times T \times d$ tensor goes through a sequence of transfomer layers, layer normalization, and a fully-connected layer.    

* The method `forward` outputs an $ n \times T \times q$ logits tensor called `H`, where $n$ is the batch size, $T$ is the number of time steps, and $q$ is the size of the vocabulary. If the tensor `H` is interpreted as a list of $n$ matrices, each of these matrices corresponds to a single sequence. Each (logits) matrix has one row per time step, so that each row contains a (transposed) logits vector.

* The method `generate` receives a string `start_text` and an integer `n_tokens` and generates a string with `len(start_text) + n_tokens` tokens by using GPT as an autoregressive language model.

* The `temperature` allows control over the sampling process. Lowering the temperature causes less probable tokens to be sampled even less frequently, and raising the temperature causes less probable tokens to be sampled more frequently.

* Note that `generate` could be implemented more efficiently by storing and reusing some intermediate results of each forward pass.


In [ ]:
class GPTLanguageModel(torch.nn.Module):
    def __init__(self, vocab, input_dimension, n_layers, n_heads, max_seq_length):
        super(GPTLanguageModel, self).__init__()

        self.vocab = vocab
        self.max_seq_length = max_seq_length

        self.token_encoding = torch.nn.Embedding(len(self.vocab), input_dimension) # The token embedding matrix
        self.position_encoding = torch.nn.Embedding(max_seq_length, input_dimension) # The position embedding matrix

        self.transformers = torch.nn.ModuleList([Transformer(input_dimension, n_heads, max_seq_length) for _ in range(n_layers)]) # List of transformer layers

        self.ln = torch.nn.LayerNorm(input_dimension) # The layer normalization
        self.fc = torch.nn.Linear(input_dimension, len(self.vocab), bias=False) # The fully-connected layer

    def forward(self, X):
        """ Keyword arguments:
        `X` -- a `batch_size` x `num_steps` matrix of token indices
        """
        tokens_encoded = self.token_encoding(X) # A `batch_size` x `num_steps` x `input_dimension` tensor
        positions_encoded = self.position_encoding(torch.arange(X.shape[1], device=X.device)) # A `num_steps` x `input_dimension` matrix

        X = tokens_encoded + positions_encoded # A `batch_size` x `num_steps` x `input_dimension` tensor
        for transformer in self.transformers:
            X = transformer(X) # A `batch_size` x `num_steps` x `input_dimension` tensor

        X = self.ln(X) # A `batch_size` x `num_steps` x `input_dimension` tensor

        H = self.fc(X) # A `batch_size` x `num_steps` x `len(self.vocab)` logits tensor

        return H

    def generate(self, start_text, n_tokens, temperature=1.0):
        """ Keyword arguments:
        `start_text`  -- a string with the initial text
        `n_tokens`    -- the number of tokens to be generated
        `temperature` -- lowering this value causes less probable tokens to be sampled even less frequently
        """
        tokens = tokenize(start_text, use_chars=True) # Converts the initial text into a list of characters
        indices = self.vocab[tokens] # Obtains the indices corresponding to the characters

        with torch.no_grad(): # Backpropagation will not be required
            for _ in range(n_tokens): # For each token to be generated
                x = indices[- self.max_seq_length:] # Truncates the list of indices to contain at most `self.max_seq_length` elements
                X = torch.tensor(x).reshape(1, -1).to(device) # Organizes the indices into a 1 x `num_steps` tensor

                output = self(X) # A `1` x `num_steps` x `len(self.vocab)` tensor

                logits = output[0, -1] # The logits vector that corresponds to the the last input vector
                p = torch.nn.functional.softmax(logits / temperature, dim=0).cpu().numpy() # Divides the logits by the temperature, applies the softmax activation function, and converts to numpy array

                index = np.random.choice(len(self.vocab), p=p) # Samples a token index according to the probabilities in `p`
                indices.append(index) # Appends the token index to the list of indices

        tokens = self.vocab.to_tokens(indices) # Converts the list of indices into a list of characters

        return ''.join(tokens) # Returns the result of concatenating the list of characters into a string

The following function can be used to initialize the parameters of each module `m` within a `GPTLanguageModel`.

In [ ]:
def init_weights(m):
    # Applies Xavier initialization if `m` is `torch.nn.Linear` or `torch.nn.Embedding`. Biases are initialized to zero.
    if type(m) == torch.nn.Linear:
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    if type(m) == torch.nn.Embedding:
        torch.nn.init.xavier_uniform_(m.weight)

# Training

* The following code defines the model hyperparameters, training hyperparameters, model, loss function, and optimizer.

In [ ]:
# Model hyperparameters
input_dimension = 64
n_layers = 4
n_heads = 8
max_seq_length = 64

# Training hyperparameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lr = 0.001 # The learning rate
num_epochs = 50 # The number of epochs for training
batch_size = 64 # The number of training sequences per batch

# Model creation
gptlm = GPTLanguageModel(vocab, input_dimension, n_layers, n_heads, max_seq_length).to(device) # Creates and moves the model to `device`
gptlm.apply(init_weights) # Applies `init_weights` to every `torch.nn.Module` inside `gptlm`

# Loss function and optimizer
loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(gptlm.parameters(), lr=lr)

* The following code implements the training loop.

In [ ]:
print(f'Using {device}.')

# Training loop
epoch_losses = []
for epoch in range(num_epochs):
    train_iter = random_partitioning(indices, batch_size, max_seq_length, offset=None)

    losses = [] # Stores the loss for each batch

    for X, Y in train_iter:
        X, Y = X.to(device), Y.to(device)
        logits = gptlm(X) # Computes the `batch_size` x `max_seq_length` x `len(vocab)` logits tensor for the `batch_size` x `max_seq_length` matrix of indices `X`

        Y_flat = Y.reshape(-1) # Flattens the `batch_size` x `max_seq_length` matrix of (target) indices `Y` into a vector with `batch_size * max_seq_length` elements
        logits_flat = logits.reshape(-1, logits.shape[2]) # Reshapes the `batch_size` x `max_seq_length` x `len(vocab)` logits tensor into a `batch_size * max_seq_length` x `len(vocab)` logits matrix

        l = loss(logits_flat, Y_flat) # Computes the loss given the logits matrix `logits_flat` and the target vector `Y_flat`

        optimizer.zero_grad() # Zeroes the gradients stored in the model parameters
        l.backward() # Computes the gradient of the loss `l` with respect to the model parameters
        optimizer.step() # Updates the model parameters based on the gradients stored inside them

        losses.append(float(l)) # Stores the loss for this batch

    epoch_losses.append(np.mean(losses))

    print(f'Epoch {epoch + 1}/{num_epochs}. Loss: {epoch_losses[-1]:.5f}.')

plt.plot(epoch_losses) # Plots the average loss across batches for each epoch
plt.xlabel('Epoch')
plt.ylabel('Average cross entropy loss')
plt.show()

# Text generation

* The following code generates text using the trained model.

* You may obtain different results by running the cell multiple times (and changing the temperature)

In [ ]:
gptlm.generate('therefore', n_tokens=512, temperature=0.5)

'therefore the delight of the passes of which i was or satched and my father had hopes and felix wretched my but i a gather day we perfect her was sleep a close of this country sat the will the same i am when i had tale been to me the most great and the door we near so friends and to the heavencess of my father which i should confessold in my ablory the sun of a shatter he resairy of the sun i desory before to the courage of his and me to quard i believe but the commage of the subject of the surprised of the first of'